# Team 14: 512-token HotpotQA retrieval baseline

Colab-first notebook: HotpotQA distractor contexts → 512-token chunks → BGE-M3 embeddings → FAISS → question top-5 chunks. Generated artifacts are written to Google Drive.

## 1. Colab setup and persistent storage

Set `GITHUB_REPO_URL` before running if the notebook is not already inside a cloned checkout. The URL is intentionally a placeholder because the team's repository URL was not provided.

In [24]:
from pathlib import Path
import os
import subprocess

GITHUB_REPO_URL = "https://github.com/amulyasagi26/adaptive-rag.git"  # Example: https://github.com/<org>/<repo>.git
REPO_NAME = "adaptive-rag"
COLAB_ROOT = Path('/content')
repo_dir = COLAB_ROOT / REPO_NAME

if GITHUB_REPO_URL and not (repo_dir / 'src').exists():
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(repo_dir)], check=True)
elif not (repo_dir / 'src').exists():
    # Supports notebooks opened from an existing checkout or uploaded with the repo.
    candidates = [Path.cwd(), Path.cwd().parent]
    repo_dir = next((p for p in candidates if (p / 'src').exists()), repo_dir)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    artifact_root = Path('/content/drive/MyDrive/team14_adaptive_rag')
except ImportError:
    artifact_root = repo_dir / 'colab_artifacts'

artifact_root.mkdir(parents=True, exist_ok=True)
print('Repository:', repo_dir)
print('Persistent artifacts:', artifact_root)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository: /content/adaptive-rag
Persistent artifacts: /content/drive/MyDrive/team14_adaptive_rag


In [25]:
# Install from the checked-out repository. This downloads packages in Colab only.
subprocess.run(['pip', 'install', '-q', '-r', str(repo_dir / 'requirements.txt')], check=True)
import sys
sys.path.insert(0, str(repo_dir))

from src.config import artifact_paths, select_device
paths = artifact_paths(artifact_root)
os.environ['HF_HOME'] = str(paths['hf_cache'])
os.environ['TRANSFORMERS_CACHE'] = str(paths['hf_cache'])
print(paths)

{'indexes': PosixPath('/content/drive/MyDrive/team14_adaptive_rag/indexes'), 'results': PosixPath('/content/drive/MyDrive/team14_adaptive_rag/results'), 'hf_cache': PosixPath('/content/drive/MyDrive/team14_adaptive_rag/hf_cache')}


In [26]:
PREFERRED_DEVICE = 'auto'  # Change to 'cpu' or 'cuda' when needed.
DEVICE = select_device(PREFERRED_DEVICE)
print('Selected device:', DEVICE)
try:
    import torch
    if DEVICE == 'cuda':
        print(torch.cuda.get_device_name(0))
except ImportError:
    pass

Selected device: cuda
Tesla T4


## 2. Load the 50-question development sample

This downloads HotpotQA into the Colab/Hugging Face cache. The corpus is pooled and deduplicated from the selected questions' distractor contexts.

In [27]:
# HotpotQA compatibility fixes
data_path = repo_dir / "src" / "data.py"
source = data_path.read_text()

# Fix 1: Hugging Face dataset identifier
source = source.replace(
    'load_dataset("hotpot_qa",',
    'load_dataset("hotpotqa/hotpot_qa",'
)

# Fix 2: Support Hugging Face context format
old_context = 'contexts = [{"title": title, "sentences": list(sentences)} for title, sentences in record["context"]]'

new_context = '''raw_context = record["context"]
    if isinstance(raw_context, dict):
        context_pairs = zip(raw_context["title"], raw_context["sentences"])
    else:
        context_pairs = raw_context

    contexts = [{"title": title, "sentences": list(sentences)} for title, sentences in context_pairs]'''

source = source.replace(old_context, new_context)

data_path.write_text(source)

# Reload the updated module
import importlib
import src.data

importlib.reload(src.data)

print("HotpotQA compatibility fixes applied.")

HotpotQA compatibility fixes applied.


In [28]:
from src.data import load_hotpotqa
from src.chunking import chunk_documents
from src.embeddings import BGEEmbedder
from src.retrieval import FaissRetriever

SAMPLE_SIZE = 50
SEED = 14
CHUNK_SIZE = 512
TOP_K = 5

examples, documents = load_hotpotqa(sample_size=SAMPLE_SIZE, seed=SEED)
print(f'{len(examples)} questions, {len(documents)} pooled documents')

50 questions, 492 pooled documents


In [29]:
# BGE-M3 is downloaded into Google Drive cache, not the Mac.
embedder = BGEEmbedder(model_name='BAAI/bge-m3', device=DEVICE)
chunks = chunk_documents(documents, embedder.tokenizer, chunk_size=CHUNK_SIZE)
embeddings = embedder.embed_chunks(chunks)
retriever = FaissRetriever.build(chunks, embeddings)
index_dir = paths['indexes'] / 'hotpotqa_512'
retriever.save(index_dir)
print(f'{len(chunks)} chunks indexed; embedding dimension={embeddings.shape[1]}')
print('FAISS index:', index_dir / 'index.faiss')
print('Metadata:', index_dir / 'metadata.json')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

492 chunks indexed; embedding dimension=1024
FAISS index: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_512/index.faiss
Metadata: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_512/metadata.json


## 3. Question → Top-5 retrieved chunks

Illustrative format only (not experiment results): an input question followed by ranked Wikipedia evidence chunks. This milestone does not generate an answer.

In [30]:
import json
import time
from src.evaluation import retrieved_token_count

question = examples[0].question
started = time.perf_counter()
results = retriever.search(question, embedder, top_k=TOP_K)
latency_ms = (time.perf_counter() - started) * 1000
print('Input question')
print(question)
print('\nRetrieved evidence')
print(f'retrieval latency: {latency_ms:.2f} ms; retrieved tokens: {retrieved_token_count([r["chunk"] for r in results])}')
for rank, result in enumerate(results, 1):
    chunk = result['chunk']
    print(f'\nChunk {rank} — {chunk.title} | Rank {rank}')
    print(f'score={result["score"]:.4f}; chunk={chunk.chunk_id}; tokens={chunk.token_count}')
    print(chunk.text[:1000])

output = {
    'question': question,
    'top_k': TOP_K,
    'latency_ms': latency_ms,
    'retrieved_token_count': retrieved_token_count([r['chunk'] for r in results]),
    'results': [
        {'score': r['score'], 'chunk': r['chunk'].__dict__} for r in results
    ],
}
output_path = paths['results'] / 'baseline_512_first_question.json'
output_path.write_text(json.dumps(output, indent=2), encoding='utf-8')
print('Saved retrieval output:', output_path)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Input question
The Alfred E. Smith Building is located across the street from the New York State Capitol and what private office building that is the thirteenth tallest structure in Albany?

Retrieved evidence
retrieval latency: 61.38 ms; retrieved tokens: 827

Chunk 1 — Alfred E. Smith Building | Rank 1
score=0.7411; chunk=8:0; tokens=223
The Alfred E. Smith Building, known officially as the Alfred E. Smith State Office Building and sometimes called simply the Smith Building, is a structure located in downtown Albany, New York across the street from the New York State Capitol and One Commerce Plaza. The building's namesake, Alfred Emmanuel Smith, was a four-term governor of New York and the Democratic Party's nomination for the 1928 presidential election. The Art Deco skyscraper has 34 stories and at 388 feet (118 m) is Albany's second tallest structure (after the Erastus Corning Tower). Completed in 1928, it houses offices of the New York State government. Built originally with an ob

## 4. Document length and chunk-size analysis

Analyze the HotpotQA development corpus using the existing BGE-M3
tokenizer and chunking function to determine whether the 256, 512,
and 1024-token configurations produce meaningful differences in
chunk count and token distribution.

In [31]:
# Analyze chunk sizes using the same development corpus

CHUNK_SIZES = [256, 512, 1024]

for size in CHUNK_SIZES:
    test_chunks = chunk_documents(
        documents,
        embedder.tokenizer,
        chunk_size=size
    )

    token_counts = [chunk.token_count for chunk in test_chunks]

    print(f"\nChunk size: {size}")
    print(f"Total documents: {len(documents)}")
    print(f"Total chunks: {len(test_chunks)}")
    print(f"Minimum tokens: {min(token_counts)}")
    print(f"Maximum tokens: {max(token_counts)}")
    print(f"Average tokens: {sum(token_counts) / len(token_counts):.2f}")
    print(f"Total tokens: {sum(token_counts)}")


Chunk size: 256
Total documents: 492
Total chunks: 536
Minimum tokens: 9
Maximum tokens: 255
Average tokens: 125.30
Total tokens: 67159

Chunk size: 512
Total documents: 492
Total chunks: 492
Minimum tokens: 13
Maximum tokens: 487
Average tokens: 136.50
Total tokens: 67159

Chunk size: 1024
Total documents: 492
Total chunks: 492
Minimum tokens: 13
Maximum tokens: 487
Average tokens: 136.50
Total tokens: 67159


## 5. Build the 256-token FAISS index

Construct a fixed-size retrieval index using 256-token chunks,
the existing HotpotQA corpus, and BGE-M3 embeddings.

In [32]:
# Build the 256-token FAISS index

chunks_256 = chunk_documents(
    documents,
    embedder.tokenizer,
    chunk_size=256
)

embeddings_256 = embedder.embed_chunks(chunks_256)

retriever_256 = FaissRetriever.build(
    chunks_256,
    embeddings_256
)

index_dir_256 = paths['indexes'] / 'hotpotqa_256'
retriever_256.save(index_dir_256)

print(f'Chunks indexed: {len(chunks_256)}')
print(f'Embedding dimension: {embeddings_256.shape[1]}')
print('FAISS index:', index_dir_256 / 'index.faiss')
print('Metadata:', index_dir_256 / 'metadata.json')

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Chunks indexed: 536
Embedding dimension: 1024
FAISS index: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_256/index.faiss
Metadata: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_256/metadata.json


## 6. Build the 1024-token FAISS index

Construct a fixed-size retrieval index using 1024-token chunks,
the existing HotpotQA corpus, and BGE-M3 embeddings.

In [33]:
# Build the 1024-token FAISS index

chunks_1024 = chunk_documents(
    documents,
    embedder.tokenizer,
    chunk_size=1024
)

embeddings_1024 = embedder.embed_chunks(chunks_1024)

retriever_1024 = FaissRetriever.build(
    chunks_1024,
    embeddings_1024
)

index_dir_1024 = paths['indexes'] / 'hotpotqa_1024'
retriever_1024.save(index_dir_1024)

print(f'Chunks indexed: {len(chunks_1024)}')
print(f'Embedding dimension: {embeddings_1024.shape[1]}')
print('FAISS index:', index_dir_1024 / 'index.faiss')
print('Metadata:', index_dir_1024 / 'metadata.json')

Batches:   0%|          | 0/31 [00:00<?, ?it/s]

Chunks indexed: 492
Embedding dimension: 1024
FAISS index: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_1024/index.faiss
Metadata: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_1024/metadata.json


## 7. Fixed-size index validation

Verify that all three FAISS indexes and their metadata have
been saved successfully. Compare the chunk contents of the
512-token and 1024-token configurations.

In [34]:
# Verify saved indexes and metadata

for size in CHUNK_SIZES:
    directory = paths['indexes'] / f'hotpotqa_{size}'

    print(f'\nChunk size: {size}')
    print('FAISS index exists:', (directory / 'index.faiss').exists())
    print('Metadata exists:', (directory / 'metadata.json').exists())

# Compare 512-token and 1024-token chunk contents

identical_chunks = (
    [chunk.text for chunk in chunks]
    ==
    [chunk.text for chunk in chunks_1024]
)

print('\n512 and 1024 chunk texts identical:', identical_chunks)


Chunk size: 256
FAISS index exists: True
Metadata exists: True

Chunk size: 512
FAISS index exists: True
Metadata exists: True

Chunk size: 1024
FAISS index exists: True
Metadata exists: True

512 and 1024 chunk texts identical: True


## 8. Comparative fixed-size retrieval experiment

Evaluate the 256, 512, and 1024-token FAISS indexes using the
same 50 HotpotQA questions. Compare supporting-fact recall,
retrieved token usage, and retrieval latency while maintaining
consistent embedding and retrieval configurations.

In [35]:
import time
import pandas as pd
from src.evaluation import supporting_fact_recall, retrieved_token_count

# Organize the three fixed-size retrievers
retrievers = {
    256: retriever_256,
    512: retriever,
    1024: retriever_1024
}

experiment_results = []

# Warm up each retriever before measuring latency
for current_retriever in retrievers.values():
    current_retriever.search(
        examples[0].question,
        embedder,
        top_k=TOP_K
    )

# Evaluate each question against all three indexes
for i, example in enumerate(examples):

    for size, current_retriever in retrievers.items():

        start_time = time.perf_counter()

        retrieved = current_retriever.search(
            example.question,
            embedder,
            top_k=TOP_K
        )

        latency_ms = (time.perf_counter() - start_time) * 1000

        retrieved_chunks = [item["chunk"] for item in retrieved]

        recall = supporting_fact_recall(
            retrieved_chunks,
            example.supporting_facts
        )

        tokens = retrieved_token_count(retrieved_chunks)

        experiment_results.append({
            "question_id": example.question_id,
            "question": example.question,
            "chunk_size": size,
            "supporting_fact_recall": recall,
            "retrieved_tokens": tokens,
            "latency_ms": latency_ms,
            "retrieved_titles": [
                chunk.title for chunk in retrieved_chunks
            ],
            "similarity_scores": [
                item["score"] for item in retrieved
            ]
        })

    if (i + 1) % 10 == 0:
        print(f"Completed {i + 1}/{len(examples)} questions")

results_df = pd.DataFrame(experiment_results)

print(f"\nTotal retrieval experiments: {len(results_df)}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (148 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Completed 50/50 questions

Total retrieval experiments: 150


In [36]:
# Calculate average retrieval metrics for each chunk size

summary_df = results_df.groupby("chunk_size").agg(
    questions_evaluated=("question_id", "count"),
    mean_recall_at_5=("supporting_fact_recall", "mean"),
    mean_retrieved_tokens=("retrieved_tokens", "mean"),
    mean_latency_ms=("latency_ms", "mean")
).reset_index()

display(summary_df.round(4))

,chunk_size,questions_evaluated,mean_recall_at_5,mean_retrieved_tokens,mean_latency_ms
0,256,50,0.925,595.20,74.9162
1,512,50,0.925,626.46,73.4567
2,1024,50,0.925,626.46,74.9779


In [37]:
# Compare recall across individual questions

recall_comparison = results_df.pivot(
    index="question_id",
    columns="chunk_size",
    values="supporting_fact_recall"
)

print("256 vs 512 recall comparison:")

print("256 performs better:",
      (recall_comparison[256] > recall_comparison[512]).sum())

print("512 performs better:",
      (recall_comparison[512] > recall_comparison[256]).sum())

print("Equal performance:",
      (recall_comparison[256] == recall_comparison[512]).sum())

print("\n512 and 1024 recall identical for all questions:",
      recall_comparison[512].equals(recall_comparison[1024]))

256 vs 512 recall comparison:
256 performs better: 0
512 performs better: 0
Equal performance: 50

512 and 1024 recall identical for all questions: True


## 9. Save comparative experiment results

Save individual retrieval results and aggregated performance
metrics to Google Drive for reproducibility, team evaluation,
and future adaptive routing experiments.

In [38]:
# Save comparative retrieval experiment results

results_dir = paths["results"]
results_dir.mkdir(parents=True, exist_ok=True)

results_df.to_json(
    results_dir / "fixed_size_retrieval_results.json",
    orient="records",
    indent=2
)

results_df.to_csv(
    results_dir / "fixed_size_retrieval_results.csv",
    index=False
)

summary_df.to_csv(
    results_dir / "fixed_size_retrieval_summary.csv",
    index=False
)

print("Experiment results saved successfully.")
print("Results directory:", results_dir)

print("\nSaved files:")
print("fixed_size_retrieval_results.json")
print("fixed_size_retrieval_results.csv")
print("fixed_size_retrieval_summary.csv")

Experiment results saved successfully.
Results directory: /content/drive/MyDrive/team14_adaptive_rag/results

Saved files:
fixed_size_retrieval_results.json
fixed_size_retrieval_results.csv
fixed_size_retrieval_summary.csv


## 11. Answer Generation Baseline

Load a lightweight instruction-tuned model (`Qwen2.5-1.5B-Instruct`) to generate natural language answers based on the retrieved context chunks.

In [42]:
# @title Load Qwen Model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Using a 1.5B parameter Qwen model
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

Loading Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully!


In [43]:
# @title Define `generate_answer` Function
def generate_answer(question, retrieved_chunks):
    # Combine the chunks into a single context string
    context = "\n\n".join([f"Document {i+1} ({chunk.title}):\n{chunk.text}" for i, chunk in enumerate(retrieved_chunks)])

    # Create the prompt using the model's chat template
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided context to answer the question briefly and accurately. If the answer is not in the context, say 'I cannot answer this based on the provided context.'"},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True
    )

    # Slice the output to only get the newly generated tokens
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [44]:
# @title Generate Answers for Sample Questions Test
import pandas as pd
import time

print("Generating answers for the first 5 questions...")
generated_results = []

# Run generation for the first 5 examples using the 512-token retriever
for i in range(5):
    example = examples[i]

    # Retrieve context using the 512-token index
    results = retriever.search(example.question, embedder, top_k=TOP_K)
    retrieved_chunks = [r['chunk'] for r in results]

    # Generate answer
    start_time = time.perf_counter()
    generated_text = generate_answer(example.question, retrieved_chunks)
    latency = (time.perf_counter() - start_time) * 1000

    generated_results.append({
        "question": example.question,
        "generated_answer": generated_text,
        "ground_truth": example.answer,
        "generation_latency_ms": round(latency, 2)
    })

# Create a DataFrame and display it
answers_df = pd.DataFrame(generated_results)
display(answers_df)

Generating answers for the first 5 questions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,question,generated_answer,ground_truth,generation_latency_ms
0,The Alfred E. Smith Building is located across...,One Commerce Plaza,One Commerce Plaza,1582.94
1,In which growing town with a population of abo...,"In the growing town of Winthrop, which had a p...","Winthrop, Maine",3452.74
2,"Rachelle Lefevre is a Canadian actress, she pl...",The Twilight Saga,The Twilight Saga,2077.46
3,Radiohead and Oceansize are bands that play wh...,"Radiohead plays rock music, while Oceansize pl...",English rock,2787.96
4,"Who has starred in more musicals, Jo Kwon or B...",Jo Kwon has starred in more musicals than Bada.,Bada,1767.63


## 12. Semantic Chunking Baseline

Unlike fixed-size chunking, **Semantic Chunking** attempts to split documents at natural topic boundaries. We achieve this by computing the embeddings of individual sentences and splitting the chunks where the cosine similarity between adjacent sentences drops below a specified threshold.

In [48]:
# @title Define and Run Semantic Chunker
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from src.chunking import Chunk
import time

def build_semantic_chunks_batched(documents, embedder, threshold=0.7, max_tokens=512):
    semantic_chunks = []

    # 1. Gather all sentences across all documents
    all_temp_chunks = []
    doc_sentence_counts = []

    for doc in documents:
        doc_sentence_counts.append(len(doc.sentences))
        for i, sentence in enumerate(doc.sentences):
            all_temp_chunks.append(Chunk(
                chunk_id=f"temp_{doc.document_id}_{i}",
                document_id=doc.document_id,
                title=doc.title,
                text=sentence,
                token_count=0,
                source_question_ids=doc.source_question_ids,
                source_sentence_indices=(i,)
            ))

    # 2. Embed all sentences at once (efficient batching)
    print(f"Embedding {len(all_temp_chunks)} sentences to analyze semantic boundaries...")
    start = time.perf_counter()
    all_embeddings = embedder.embed_chunks(all_temp_chunks)
    print(f"Finished sentence embedding in {time.perf_counter() - start:.2f} seconds.")

    # 3. Process document by document
    current_idx = 0
    for doc, num_sentences in zip(documents, doc_sentence_counts):
        if num_sentences == 0:
            continue

        doc_embeddings = all_embeddings[current_idx : current_idx + num_sentences]
        current_idx += num_sentences

        if num_sentences == 1:
            text = doc.sentences[0]
            tokens = len(embedder.tokenizer.encode(text))
            semantic_chunks.append(Chunk(
                chunk_id=f"{doc.document_id}:0",
                document_id=doc.document_id,
                title=doc.title,
                text=text,
                token_count=tokens,
                source_question_ids=doc.source_question_ids,
                source_sentence_indices=(0,)
            ))
            continue

        # Calculate similarities between adjacent sentences
        similarities = []
        for i in range(num_sentences - 1):
            sim = cosine_similarity([doc_embeddings[i]], [doc_embeddings[i+1]])[0][0]
            similarities.append(sim)

        # Identify split points (similarity drops below threshold)
        split_indices = set(i + 1 for i, sim in enumerate(similarities) if sim < threshold)

        # Group into chunks
        current_sentences = []
        current_indices = []
        current_tokens = 0
        chunk_idx = 0

        for i, sentence in enumerate(doc.sentences):
            sentence_tokens = len(embedder.tokenizer.encode(sentence))

            # Split if semantic boundary is hit or max tokens exceeded
            if (i in split_indices and current_sentences) or (current_tokens + sentence_tokens > max_tokens):
                text = " ".join(current_sentences)
                semantic_chunks.append(Chunk(
                    chunk_id=f"{doc.document_id}:{chunk_idx}",
                    document_id=doc.document_id,
                    title=doc.title,
                    text=text,
                    token_count=current_tokens,
                    source_question_ids=doc.source_question_ids,
                    source_sentence_indices=tuple(current_indices)
                ))
                current_sentences = []
                current_indices = []
                current_tokens = 0
                chunk_idx += 1

            current_sentences.append(sentence)
            current_indices.append(i)
            current_tokens += sentence_tokens

        # Add remaining sentences
        if current_sentences:
            text = " ".join(current_sentences)
            semantic_chunks.append(Chunk(
                chunk_id=f"{doc.document_id}:{chunk_idx}",
                document_id=doc.document_id,
                title=doc.title,
                text=text,
                token_count=current_tokens,
                source_question_ids=doc.source_question_ids,
                source_sentence_indices=tuple(current_indices)
            ))

    return semantic_chunks

# Execute semantic chunking
# A threshold of 0.7 is a good starting point for BGE embeddings.
semantic_chunks = build_semantic_chunks_batched(documents, embedder, threshold=0.7, max_tokens=512)

token_counts = [c.token_count for c in semantic_chunks]
print(f"\nGenerated {len(semantic_chunks)} semantic chunks.")
print(f"Average tokens per chunk: {sum(token_counts)/len(token_counts):.2f}")
print(f"Min tokens: {min(token_counts)}, Max tokens: {max(token_counts)}")


Embedding 2074 sentences to analyze semantic boundaries...


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Finished sentence embedding in 22.26 seconds.

Generated 2000 semantic chunks.
Average tokens per chunk: 35.65
Min tokens: 4, Max tokens: 198


In [49]:
# @title Build FAISS Index for Semantic Chunks
from src.retrieval import FaissRetriever

print("Embedding semantic chunks for the FAISS index...")
semantic_embeddings = embedder.embed_chunks(semantic_chunks)

# Build and save
semantic_retriever = FaissRetriever.build(semantic_chunks, semantic_embeddings)
semantic_index_dir = paths['indexes'] / 'hotpotqa_semantic'
semantic_retriever.save(semantic_index_dir)

print(f'FAISS index saved to: {semantic_index_dir}')
print(f'Indexed {len(semantic_chunks)} chunks with dimension {semantic_embeddings.shape[1]}')


Embedding semantic chunks for the FAISS index...


Batches:   0%|          | 0/125 [00:00<?, ?it/s]

FAISS index saved to: /content/drive/MyDrive/team14_adaptive_rag/indexes/hotpotqa_semantic
Indexed 2000 chunks with dimension 1024


In [51]:
# @title Evaluate Semantic Retrieval vs Fixed-Size
from src.evaluation import supporting_fact_recall, retrieved_token_count
import pandas as pd
import time

print("Evaluating retrieval performance of semantic chunks across different Top-K values...")
top_k_options = [5, 10, 15]
semantic_results = []

for k in top_k_options:
    for example in examples:
        start_time = time.perf_counter()
        retrieved = semantic_retriever.search(example.question, embedder, top_k=k)
        latency_ms = (time.perf_counter() - start_time) * 1000

        retrieved_chunks = [item["chunk"] for item in retrieved]
        recall = supporting_fact_recall(retrieved_chunks, example.supporting_facts)
        tokens = retrieved_token_count(retrieved_chunks)

        semantic_results.append({
            "question_id": example.question_id,
            "top_k": k,
            "chunk_size": "semantic",
            "supporting_fact_recall": recall,
            "retrieved_tokens": tokens,
            "latency_ms": latency_ms
        })

semantic_df = pd.DataFrame(semantic_results)

# Aggregate metrics
semantic_summary = semantic_df.groupby(["chunk_size", "top_k"]).agg(
    questions_evaluated=("question_id", "count"),
    mean_recall_at_k=("supporting_fact_recall", "mean"),
    mean_retrieved_tokens=("retrieved_tokens", "mean"),
    mean_latency_ms=("latency_ms", "mean")
).reset_index()

print("\n--- Semantic Chunking Performance (Varying Top-K) ---")
display(semantic_summary.round(4))

print("\n--- Previous Fixed-Size Baseline Comparison (Top-5) ---")
display(summary_df.round(4))

Evaluating retrieval performance of semantic chunks across different Top-K values...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (145 outputs left out)


--- Semantic Chunking Performance (Varying Top-K) ---


,chunk_size,top_k,questions_evaluated,mean_recall_at_k,mean_retrieved_tokens,mean_latency_ms
0,semantic,5,50,0.780,176.92,50.5310
1,semantic,10,50,0.885,356.86,53.4323
2,semantic,15,50,0.940,536.28,53.6534



--- Previous Fixed-Size Baseline Comparison (Top-5) ---


,chunk_size,questions_evaluated,mean_recall_at_5,mean_retrieved_tokens,mean_latency_ms
0,256,50,0.925,595.20,74.9162
1,512,50,0.925,626.46,73.4567
2,1024,50,0.925,626.46,74.9779


In [52]:
# @title Evaluate Semantic Retrieval with Higher Top-K
print("Testing Semantic Chunking with higher top_k values...")
top_k_options = [10, 15]
extended_results = []

for k in top_k_options:
    for example in examples:
        start_time = time.perf_counter()
        # Search with the new top_k value
        retrieved = semantic_retriever.search(example.question, embedder, top_k=k)
        latency_ms = (time.perf_counter() - start_time) * 1000

        retrieved_chunks = [item["chunk"] for item in retrieved]
        recall = supporting_fact_recall(retrieved_chunks, example.supporting_facts)
        tokens = retrieved_token_count(retrieved_chunks)

        extended_results.append({
            "question_id": example.question_id,
            "top_k": k,
            "supporting_fact_recall": recall,
            "retrieved_tokens": tokens,
            "latency_ms": latency_ms
        })

extended_df = pd.DataFrame(extended_results)

# Aggregate metrics to see how much better we do!
extended_summary = extended_df.groupby("top_k").agg(
    questions_evaluated=("question_id", "count"),
    mean_recall=("supporting_fact_recall", "mean"),
    mean_retrieved_tokens=("retrieved_tokens", "mean"),
    mean_latency_ms=("latency_ms", "mean")
).reset_index()

print("\n--- Semantic Chunking Performance (Higher Top-K) ---")
display(extended_summary.round(4))

Testing Semantic Chunking with higher top_k values...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

... (93 outputs left out)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Semantic Chunking Performance (Higher Top-K) ---


,top_k,questions_evaluated,mean_recall,mean_retrieved_tokens,mean_latency_ms
0,10,50,0.885,356.86,54.9275
1,15,50,0.940,536.28,55.7045
